In [1]:
import pandas as pd
from datasets import load_from_disk
from src.data import DATA_DIR_RAW

In [2]:
from ir_datasets_longeval import load

In [ ]:
dataset = load("longeval-2023/2022-06/en")

# LongEval
- To use the click model judgments as context to generate the topics they need to be somewhat reliable. How do the human judgments align with the click model judgments? 

In [3]:
BASEPATH = DATA_DIR_RAW / "datasets" / "long_eval_ictir2025"

In [4]:
def ds_to_qrels(ds_path: str):
    """Load Huggingface Datasets dataset and transform it to the TREC qrels format"""
    dataset = load_from_disk(ds_path).to_pandas()
    dataset["Q0"] = "0"
    return dataset[["query_id", "Q0", "doc_id", "relevance"]]

In [5]:
qrels_deep_31 = ds_to_qrels(BASEPATH / "depth_based_31_50_50")
qrels_deep_45 = ds_to_qrels(BASEPATH / "depth_based_45_25_25")

In [13]:
df = load_from_disk(BASEPATH / "depth_based_45_25_25").to_pandas()

In [14]:
df["query_id"] = df["query_id"].apply(lambda x: dataset.query_id_map[x])
df["doc_id"] = df["doc_id"].apply(lambda x: dataset.doc_id_map[x])

In [9]:
queries = pd.DataFrame(dataset.queries)

In [18]:
df = df.merge(queries, left_on="query_id", right_on="query_id")

In [23]:
store = dataset.docs_store()

In [24]:
df["text_my"] = df["doc_id"].apply(lambda x: store.get(x).text)

In [25]:
df

,query_id,doc_id,query_text,doc_text,relevance,text,text_my
0,401ad27c-2b5d-4f2d-9d92-2b1768b779e4,1f57ccc2-120a-4b57-85e7-cda70a2fe3cc,space sfr,"Localiser ip 79.83.119.10, the address is loca...",1,space sfr,"\n\nLocaliser ip 79.83.119.10, the address is ..."
1,401ad27c-2b5d-4f2d-9d92-2b1768b779e4,068d4d73-e845-49b7-a8f6-cbfc6bf8fe09,space sfr,U'GO\n- SFR for Android\nU'\nGO\n- SFR for And...,1,space sfr,\n\nU'GO\n- SFR for Android\nU'\nGO\n- SFR for...
2,401ad27c-2b5d-4f2d-9d92-2b1768b779e4,c5e24ec0-ad87-4396-b00f-6b0434f0d19a,space sfr,monespace.sfr.re.\n- my space Monespace.sfr.re...,1,space sfr,\n\nmonespace.sfr.re.\n- my space Monespace.sf...
3,401ad27c-2b5d-4f2d-9d92-2b1768b779e4,ff946b20-7135-4666-b431-6e8f7b6944e2,space sfr,SFR Video Club : SFR VoD - Select your program...,1,space sfr,\n\nSFR Video Club : SFR VoD - Select your pro...
4,401ad27c-2b5d-4f2d-9d92-2b1768b779e4,0cc014ad-9ffa-4b02-a47e-84177b2ffdc4,space sfr,rsqynucgmu'\ns Journal\n--For use in the manuf...,1,space sfr,\n\nrsqynucgmu'\ns Journal\n--For use in the m...
...,...,...,...,...,...,...,...
2245,73f598c6-c319-4697-8e8c-f8043a261fc0,9192ae95-5388-48b5-ac31-9c727f02af94,car rental nants,Theft\nParis Group\nBudapest Easyjet cheap | T...,0,car rental nants,\n\nTheft\nParis Group\nBudapest Easyjet cheap...
2246,73f598c6-c319-4697-8e8c-f8043a261fc0,b21a50ca-8549-437c-8dac-79f911695116,car rental nants,Rental Utility\nPeugeot\nBoxing 2015\nDiesel i...,0,car rental nants,\n\ndoc062201806909\n\nRental Utility\nPeugeot...
2247,73f598c6-c319-4697-8e8c-f8043a261fc0,70732c48-e063-4ac7-8e19-03fe173ac875,car rental nants,Car hire in Saumur from 18€/d\n- Getaround (ex...,0,car rental nants,\n\nCar hire in Saumur from 18€/d\n- Getaround...
2248,73f598c6-c319-4697-8e8c-f8043a261fc0,cd7f9458-a2aa-4e2a-ad0b-7cb012b080f5,car rental nants,Car Rental in La Trinité-sur-Mer\nfrom 18€/d\n...,0,car rental nants,\n\nCar Rental in La Trinité-sur-Mer\nfrom 18€...


In [7]:
len(qrels_deep_31)

3100

### Duplicate Qrels

In [8]:
print("Duplecates in dataset:", len(qrels_deep_31) -
      len(qrels_deep_31.drop_duplicates(["query_id", "doc_id"])))
print("Duplecates in dataset:", len(qrels_deep_45) -
      len(qrels_deep_45.drop_duplicates(["query_id", "doc_id"])))

Duplecates in dataset: 180
Duplecates in dataset: 58


In [9]:
print("Duplecates in dataset:", len(qrels_deep_31) -
      len(qrels_deep_31.drop_duplicates(["query_id", "doc_id", "relevance"])))
print("Duplecates in dataset:", len(qrels_deep_45) -
      len(qrels_deep_45.drop_duplicates(["query_id", "doc_id", "relevance"])))

Duplecates in dataset: 52
Duplecates in dataset: 13


### How well aligne the human and click model judgments?
Small overlap: 
- 31 Topics: Shallow and deep have 134 overlapping qrels
- 45 Topics: Shallow and deep have 99 overlapping qrels

Insufficient correlation:
- 31 Topics: 0.07
- 45 Topics: 0.06

In [8]:
from ir_datasets_longeval import load
from sklearn.metrics import cohen_kappa_score

In [9]:
dataset = load("longeval-2023/2022-06/en/non-unified")
qrels = pd.DataFrame(dataset.qrels)

In [10]:
qrels["relevance"] = qrels["relevance"].apply(lambda x: 1 if x >= 1 else 0)

In [11]:
df = qrels.merge(qrels_deep_31, how="inner", left_on=[
                 "query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [12]:
print("Shallow and deep have", len(df), "overlapping qrels")

Shallow and deep have 134 overlapping qrels


In [13]:
cohen_kappa_score(df["relevance_x"].astype(str), df["relevance_y"].astype(str))

0.06749285033365093

In [14]:
df = qrels.merge(qrels_deep_45, how="inner", left_on=[
                 "query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [15]:
print("Shallow and deep have", len(df), "overlapping qrels")

Shallow and deep have 99 overlapping qrels


In [16]:
cohen_kappa_score(df["relevance_x"].astype(str), df["relevance_y"].astype(str))

0.0551053484602918

# Overlap

#### 31 Queries

In [31]:
df = qrels.merge(qrels_deep_31, how="left", left_on=["query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [32]:
# click model judgment not judgeed by humans
train_split = df[(df["relevance_y"].isna()) & (df["query_id"].isin(qrels_deep_31["query_id"]))]

In [33]:
# All queries are judged
train_split["query_id"].nunique()

31

In [34]:
# at least 4 judgments per query not judged by humans
train_split.groupby("query_id")["relevance_x"].count().describe()

count    31.000000
mean     11.000000
std       7.056912
min       4.000000
25%       8.000000
50%       9.000000
75%      10.500000
max      35.000000
Name: relevance_x, dtype: float64

In [35]:
train_split[train_split["relevance_x"] == 0].groupby("query_id")["relevance_x"].count().min()

np.int64(2)

In [36]:
train_split[train_split["relevance_x"] >= 1].groupby("query_id")["relevance_x"].count().min()

np.int64(1)

#### 45 Queries

In [25]:
df = qrels.merge(qrels_deep_45, how="left", left_on=["query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [26]:
# click model judgment not judgeed by humans
train_split = df[(df["relevance_y"].isna()) & (df["query_id"].isin(qrels_deep_45["query_id"]))]

In [27]:
# All queries are judged
train_split["query_id"].nunique()

45

In [ ]:
# at least 8 judgments per query not judged by humans
train_split.groupby("query_id")["relevance_x"].count().describe()

count    45.000000
mean     12.311111
std       6.639490
min       8.000000
25%       9.000000
50%      10.000000
75%      12.000000
max      43.000000
Name: relevance_x, dtype: float64

In [29]:
train_split[train_split["relevance_x"] == 0].groupby("query_id")["relevance_x"].count().min()

np.int64(4)

In [30]:
train_split[train_split["relevance_x"] >= 1].groupby("query_id")["relevance_x"].count().min()

np.int64(1)